# 11 — **헛알림까지 포함한** 커버리지를 전체 데이터로 (STEP 31)

## 묻는 것 하나

STEP 27~30 은 *"확신 있을 때만 병변 이름을 말하면 몇 %나 말할 수 있나"* 에
답했습니다. 그런데 **VL01(전체의 10.8%)에서만** 쟀습니다.

| | VL01 에서 잰 값 | 전체에서는? |
|---|---:|---|
| 6종 이름 | 33.7% | ? |
| 계열 4군 | 61.4% | ? |
| A6 경보 (정밀도 90%) | 재현율 12.1% | **50.4%** ← 이미 확인, 4배 차이 |

**A6 경보에서 VL01 이 4배 넘게 깎아 보고 있었습니다.** 커버리지에서도 VL01 은
비관적이라(STEP 25), 전체에서는 더 높을 가능성이 큽니다. 그런데 STEP 23·25 는
VL01 이 **낙관적**이었던 사례라 — **방향을 짐작하지 말고 재야 합니다.**

## 왜 로컬에서 못 하나

로컬엔 **VL01 크롭만** 있습니다. TL01·TL02 크롭은 캐글에 있고, 원본을 이 PC 로
받는 건 물리적으로 불가능합니다 (TL02 250GB > 디스크 238GB).

## 학습이 **없습니다** — 배열 하나만 만듭니다

    p1        1단계 '이상' 확률 (온도 보정 적용)
    truth     실제 라벨 (A1~A6, A7=정상)
    p2 × 3팔  1단계가 넘긴 사진에 대한 2단계 확률

이 배열만 있으면 알갱이·하한선·A6 경보 분석이 **전부 로컬에서 공짜로** 돕니다.
`tools/naming_granularity.py` · `tools/naming_baseline.py` 가 그대로 읽습니다.

## 붙일 것 (Add Input)

| 입력 | 왜 |
|---|---|
| `m2.5` 크롭 · `f320` 크롭 · 매니페스트 | 노트북 10 과 같습니다 |
| **STEP 16 release** | 1단계 + 2단계 릴리스 |
| **STEP 23 노트북 Output** | `stage2_effnetv2_s_{m2.5,f320}_384_moderate` |

⚠️ 노트북 10 과 **똑같은 입력**입니다. 그때 붙인 걸 그대로 쓰세요.

## 돌리는 법

우측 상단 **[Save Version] → Save & Run All (Commit)**. 1~2시간.


In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
NAME   = "deeplearning_test"
# ⚠️ 브랜치를 "main" 으로 **못 박으면 안 됩니다.** 아래 reset --hard 가
#    작업 브랜치를 통째로 덮어써서, 방금 만든 코드가 사라진 채로 몇 시간을
#    돌게 됩니다. 이미 리포 안에서 돌고 있으면 **지금 브랜치를 그대로 씁니다.**
#    바꾸려면 환경변수:  export DOG_SKIN_BRANCH=main
# ★ 이 노트북이 사는 브랜치. **여기서 못 박지 않으면 "main" 을 받습니다.**
#    캐글/콜랩은 리포가 없는 상태로 시작해서 아래 _ROOT 탐색이 실패하고,
#    예전 기본값이 "main" 이었습니다. main 이 뒤처져 있으면 **셀은 최신인데
#    src/ 만 옛것**인 채로 돕니다 — 실제로 며칠 그랬습니다 (main 75445c0).
#    첫 셀은 그 상태에서도 "코드 버전 …" 을 태연히 찍습니다.
NB_BRANCH = "claude/dog-disease-diagnosis-model-1s6jtf"
BRANCH = os.environ.get("DOG_SKIN_BRANCH", "")
_cwd   = os.getcwd()

# ⚠️ "지금 리포 안인가" 를 **폴더 이름으로만** 보면 안 됩니다. 주피터에서
#    notebooks/*.ipynb 를 열면 cwd 가 `.../deeplearning_test/notebooks` 라
#    이름이 안 맞고, 그러면 **리포 안에 리포를 또 clone** 합니다
#    (실제로 런팟에서 .../notebooks/deeplearning_test 가 생겼습니다).
#    위로 거슬러 올라가며 **진짜 리포 루트**를 찾습니다.
_p = os.path.abspath(_cwd)
_ROOT = None
while True:
    if (os.path.isdir(os.path.join(_p, ".git"))
            and os.path.isfile(os.path.join(_p, "src", "env.py"))):
        _ROOT = _p
        break
    _up = os.path.dirname(_p)
    if _up == _p:
        break
    _p = _up

if _ROOT:
    DIR = _ROOT           # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
    if not BRANCH:
        BRANCH = subprocess.run(["git", "-C", DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                                capture_output=True, text=True).stdout.strip() or "main"
else:
    # ⚠️ Kaggle 을 먼저 봅니다. Kaggle 이미지에도 /content 가 있어서
    #    /content 를 먼저 보면 Kaggle 세션인데 /content 에 clone 합니다.
    BASE = ("/kaggle/working" if os.path.isdir("/kaggle/working")
            else "/content" if os.path.isdir("/content") else _cwd)
    DIR = os.path.join(BASE, NAME)

BRANCH = BRANCH or NB_BRANCH

# ⚠️ 예전엔 fetch/reset 을 **둘 다 check=False** 로 불렀습니다. 실패해도 조용히
#    넘어가서, 캐글 클론이 **지워진 커밋(75445c0)에 붙박인 채 며칠을 돌았습니다.**
#    src/ 를 아무리 고쳐 푸시해도 안 실렸고, 첫 셀은 "코드 버전 …" 을 태연히
#    찍었습니다. 그 줄을 믿을 수 없다는 게 제일 나빴습니다.
#    → 이제 실패하면 **말하고, 클론을 지우고 다시 받습니다.**
#    (Kaggle Persistence 를 'Files' 로 켜두면 /kaggle/working 이 살아남아
#     낡은 클론이 계속 재사용됩니다 — 그 경우에도 여기서 복구됩니다.)
def _git(*args, cwd=None):
    return subprocess.run(["git", *args], capture_output=True, text=True, cwd=cwd)


def _fresh_clone(dst, branch):
    import shutil as _sh
    _sh.rmtree(dst, ignore_errors=True)
    r = _git("clone", "-b", branch, "--depth", "1", REPO, dst)
    if r.returncode != 0:
        raise RuntimeError("git clone 실패:\n" + (r.stderr or "")[-800:])


_need_clone = not os.path.isdir(os.path.join(DIR, ".git"))
if not _need_clone:
    # shallow clone 이라 origin/<브랜치> 대신 FETCH_HEAD 로 맞춥니다
    # (히스토리가 갈리면 origin/<브랜치> 가 옛 커밋을 가리킨 채 남습니다)
    r = _git("-C", DIR, "fetch", "--depth", "1", "origin", BRANCH)
    if r.returncode != 0:
        print("⚠️ git fetch 실패 — 클론을 새로 받습니다\n   " + (r.stderr or "")[-300:])
        _need_clone = True
    else:
        r = _git("-C", DIR, "reset", "--hard", "FETCH_HEAD")
        if r.returncode != 0:
            print("⚠️ git reset 실패 — 클론을 새로 받습니다\n   " + (r.stderr or "")[-300:])
            _need_clone = True

if _need_clone:
    _fresh_clone(DIR, BRANCH)

# ★ 정말 최신인지 **확인**합니다. 위가 다 성공해도 여기서 한 번 더 봅니다 —
#   "최신이라고 믿었는데 아니었다" 가 이 프로젝트에서 가장 비쌌던 실패입니다.
_local = _git("-C", DIR, "rev-parse", "HEAD").stdout.strip()
_remote = _git("-C", DIR, "ls-remote", REPO, f"refs/heads/{BRANCH}").stdout.split()
_remote = _remote[0] if _remote else ""
if _remote and _local and not _remote.startswith(_local[:8]) and not _local.startswith(_remote[:8]):
    print("\n" + "!" * 66)
    print(f"🚨 코드가 최신이 아닙니다 — 로컬 {_local[:8]} / 원격 {_remote[:8]}")
    print("   클론을 지우고 다시 받습니다.")
    print("!" * 66 + "\n")
    _fresh_clone(DIR, BRANCH)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", _git("-C", DIR, "log", "--oneline", "-1").stdout.strip())
print("브랜치      :", BRANCH,
      f"(원격 {_remote[:8]})" if _remote else "(원격 확인 실패)")
if BRANCH != NB_BRANCH:
    print(f"⚠️ 이 노트북이 만들어진 브랜치({NB_BRANCH})가 아닙니다 —")
    print("   src/ 가 셀보다 뒤처져 있을 수 있습니다. 아래 [nb] 줄을 꼭 보세요.")

# 패키지 설치는 **uv 로 통일**합니다 (pip 보다 훨씬 빠릅니다).
# ⚠️ Colab/Kaggle 이미지에는 uv 가 없어서, uv 자체만 pip 로 한 번 받습니다.
#    --system = 가상환경을 새로 만들지 않고 이미 있는 파이썬에 그대로 설치.
#    (torch/numpy/pandas 는 이미 깔려 있으므로 여기서 안 건드립니다)
# albumentations 는 import 할 때마다 PyPI 에 버전 확인 요청을 보냅니다.
# Kaggle 은 외부 네트워크가 막혀 있어 타임아웃(2초)만 기다리다 끝납니다 — 꺼둡니다.
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

# ⚠️ 임대 GPU 이미지(런팟 등)의 파이썬은 **externally managed** 입니다 (PEP 668).
#    그냥 설치하면 첫 시도가 통째로 거부돼서, 재시도 로직이 있어도 무서운
#    에러 덩어리가 먼저 찍힙니다. 처음부터 허용해두면 그 소음이 없습니다.
#    Colab/Kaggle 에는 이 제약이 없어서 이 변수는 무해합니다.
os.environ["PIP_BREAK_SYSTEM_PACKAGES"] = "1"
os.environ["UV_BREAK_SYSTEM_PACKAGES"] = "1"

# ⚠️ Colab/Kaggle 에는 numpy·pandas·sklearn 이 이미 있지만 **임대 GPU 이미지엔
#    torch 만 있는 경우가 많습니다** (런팟에서 `No module named 'pandas'` 로
#    막혔습니다). 그렇다고 매번 다 깔면 Colab 에서 버전이 흔들리므로
#    **없는 것만** 깝니다.
_NEED = {                       # import 이름 → pip 이름
    "numpy": "numpy", "pandas": "pandas", "pyarrow": "pyarrow", "PIL": "Pillow",
    "sklearn": "scikit-learn", "cv2": "opencv-python-headless", "tqdm": "tqdm",
    "matplotlib": "matplotlib", "timm": "timm", "imagehash": "imagehash",
    "pytorch_grad_cam": "grad-cam", "albumentations": "albumentations",
}
import importlib.util as _ilu

_PKGS = [pip for mod, pip in _NEED.items() if _ilu.find_spec(mod) is None]
if _PKGS:
    print(f"[env] 없는 패키지 {len(_PKGS)}개를 깝니다: {_PKGS}")
else:
    print("[env] 필요한 패키지가 전부 있습니다 — 설치를 건너뜁니다")

# ⚠️ 일부 이미지(런팟 PyTorch 등)는 파이썬이 **externally managed** 라
#    (PEP 668) --system 설치를 거부합니다. Colab/Kaggle 에는 없는 문제라
#    처음엔 안 넣었다가 런팟에서 첫 셀이 바로 죽었습니다.
#    --break-system-packages 를 붙여 한 번 더 시도합니다.
def _install(args: list[str]) -> bool:
    return subprocess.run(args, check=False).returncode == 0


_ok = not _PKGS          # 깔 게 없으면 이미 성공입니다
if _PKGS and _install([sys.executable, "-m", "pip", "install", "-q", "uv"]):
    _base = [sys.executable, "-m", "uv", "pip", "install", "-q", "--system"]
    _ok = _install(_base + _PKGS)
    if not _ok:
        _ok = _install(_base + ["--break-system-packages"] + _PKGS)
if not _ok:
    print("[env] uv 로 설치하지 못해 pip 으로 대체합니다")
    _p = [sys.executable, "-m", "pip", "install", "-q"]
    if not _install(_p + _PKGS):
        _install(_p + ["--break-system-packages"] + _PKGS)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

MY_NOTEBOOK_VERSION = "2026-09-04.5"   # ★ 이 셀(=이 .ipynb)의 버전

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

# 환경 판정이 이상하면(예: Kaggle 인데 colab 이라고 나오면) 근거를 봅니다
if E.env != "local":
    env.diagnose()

# ⚠️ 노트북 셀은 git pull 로 갱신되지 않습니다 (src/ 만 최신이 됩니다).
#    낡은 .ipynb 를 몇 시간 돌리고 나서 알게 되면 늦으므로 지금 확인합니다.
from src.config import NOTEBOOK_VERSION as _repo_nb
if MY_NOTEBOOK_VERSION != _repo_nb:
    print("\n" + "!" * 62)
    print(f"⚠️ 이 노트북이 낡았습니다 — 내 셀 {MY_NOTEBOOK_VERSION} / 리포 {_repo_nb}")
    print("   src/ 는 최신이지만 **셀 내용은 예전 것**입니다.")
    print("   GitHub 에서 notebooks/*.ipynb 를 다시 받아 Import 하세요:")
    print("   Kaggle → File → Import Notebook / Colab → 파일 → 노트 업로드")
    print("!" * 62 + "\n")
else:
    print(f"[nb] 노트북 최신 ({_repo_nb})")


## 1. 체크포인트 먼저 — **크롭 연결보다 앞에서** 멈춥니다

노트북 10 첫 판이 크롭 연결 24분을 다 태우고 "체크포인트가 없습니다" 로
죽었습니다. 1초짜리 검사를 24분 뒤에 알려준 셈이라, 순서를 바꿨습니다.


In [ ]:
import sys
sys.path.insert(0, DIR)
import json, time
import numpy as np, torch
from pathlib import Path
from src import crop, data, env, labels, models, split, stages, train
from src.config import CFG, CLASSES

STAGE1 = "stage1_effnetv2_s_f320_384_n233k_moderate_photometric"
ARMS = [
    ("m2.5(cnx릴리스)", "stage2_convnextv2_base_m2.5_384_n121k_moderate", "m2.5"),
    ("m2.5(eff)",      "stage2_effnetv2_s_m2.5_384_moderate",            "m2.5"),
    ("f320(eff)",      "stage2_effnetv2_s_f320_384_moderate",            "f320"),
]

# ★ 붙인 데이터셋은 /kaggle/input 에 있습니다 — 작업 폴더가 아닙니다.
train.import_previous_run()

ck = env.work_root() / "checkpoints"
need = [STAGE1] + [e for _, e, _ in ARMS]
missing = [e for e in need if not (ck / e / "best.pt").exists()]
if missing:
    have = sorted(p.name for p in ck.glob("*") if (p / "best.pt").exists())
    msg = ["[X] 체크포인트가 없습니다: " + ", ".join(missing),
           f"    작업 폴더({ck})에 있는 것:"]
    msg += [f"      {h}" for h in (have or ["(없음)"])]
    srcs = train.find_checkpoint_sources()
    msg.append("    붙어 있는 입력에서 찾은 checkpoints 폴더:")
    if srcs:
        for s in srcs:
            msg.append(f"      {s}")
            msg += [f"        - {p.parent.name}" for p in sorted(s.glob("*/best.pt"))]
    else:
        msg.append("      (하나도 없음)")
    inp = Path("/kaggle/input")
    if inp.is_dir():
        msg.append("    /kaggle/input 안:")
        for a in sorted(inp.iterdir()):
            msg.append(f"      {a.name}/")
    raise SystemExit("\n".join(msg))

# 임계값도 **여기서** 확인합니다 (없으면 recall 이 조용히 무너집니다)
thr_file = next((p for p in [env.work_root() / "stage1_threshold.json",
                             *env.work_root().glob("**/stage1_threshold.json")]
                 if p.exists()), None)
if thr_file is None:
    raise SystemExit("[X] stage1_threshold.json 이 없습니다 — 기본값을 쓰면 안 됩니다")
THR = float(json.loads(thr_file.read_text(encoding="utf-8"))["threshold"])
print(f"체크포인트 4개 확인 · 1단계 임계값 {THR:.4f}  ({thr_file})")

# 크롭 연결은 확인이 끝난 뒤 (20~60분, GPU 0% 가 정상)
env.load_prepared()
env.require_gpu()
DEV = "cuda"


## 2. 1단계 val — **정상 사진까지 전부**

여기가 STEP 27 의 핵심이었습니다. 화면에 뜨는 건 *진짜 병변인 사진*이 아니라
**1단계가 "이상" 으로 넘긴 사진 전부**이고, 거기엔 멀쩡한 개가 섞여 있습니다.
그 사진에 병변 이름이 붙으면 **무조건 오답**입니다.


In [ ]:
df = labels.load(env.work_root() / "manifests" / "manifest_final.parquet")
TAGS = sorted({t for _, _, t in ARMS} | {"f320"})
have = crop.available_tags()
print(f"{len(df):,}행 · 붙어 있는 태그 {have}")
if [t for t in TAGS if t not in have]:
    raise SystemExit(f"[X] 태그 없음: {[t for t in TAGS if t not in have]}")

keep = crop.chunks_with_crops(df, TAGS)
if not keep:
    raise SystemExit("[X] 필요한 태그가 다 있는 청크가 없습니다.")
df = df[df["chunk"].isin(keep)].reset_index(drop=True)
print(f"쓸 청크 {keep} — {len(df):,}행")

# 1단계 뷰: 정상 + 병변 **전부** (f320 크롭)
s1 = stages.to_stage1(crop.switch_tag(df, "f320", verbose=False), verbose=False)
_, va = split.get_fold(s1, 0)
va = va.reset_index(drop=True)
y1 = (va["label"] != stages.NORMAL_LABEL).to_numpy().astype(int)
print(f"1단계 val {len(va):,}장  (정상 {int((y1 == 0).sum()):,} / "
      f"병변 {int(y1.sum()):,})")


## 3. 1단계 추론 → 넘긴 사진 고르기

온도 보정을 **적용합니다** — 서빙(`infer.Engine`)이 `softmax(logit / T)` 를
돌려주므로, 여기서 안 걸면 서빙과 다른 확률로 판단하게 됩니다.


In [ ]:
def temperature(exp):
    f = train.ckpt_dir(exp) / "temperature.json"
    if not f.exists():
        print(f"  ⚠️ {exp}/temperature.json 없음 — T=1.0")
        return 1.0
    return float(json.loads(f.read_text(encoding="utf-8"))["temperature"])


def softmax(a):
    e = np.exp(a - a.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def infer(exp, frame, classes):
    """이 실험의 모델로 frame 전체를 추론 → 로짓.

    ⚠️ 반드시 `data.eval_loader` — 행 순서가 보존돼야 1·2단계를 짝지을 수
       있습니다. 직접 만든 로더로 갈라졌던 적이 있습니다.
    """
    m = models.build(train.model_key_from_exp(exp), n_classes=len(classes),
                     pretrained=False)
    sd = torch.load(train.ckpt_dir(exp) / "best.pt", map_location="cpu",
                    weights_only=False)
    m.load_state_dict(sd.get("model", sd.get("state_dict", sd)), strict=False)
    m = m.to(DEV).eval()
    cfg = CFG(); cfg.img_size = 384
    dl, ds = data.eval_loader(frame, cfg, model=m, classes=classes)
    if len(ds) != len(frame):
        raise SystemExit(f"[X] {exp}: {len(frame) - len(ds):,}행이 빠졌습니다 "
                         "— 행이 어긋나면 짝짓기가 조용히 망가집니다")
    out = []
    with torch.no_grad():
        for x, _ in dl:
            out.append(m(x.to(DEV)).float().cpu())
    del m; torch.cuda.empty_cache()
    return torch.cat(out).numpy()


t0 = time.time()
T1 = temperature(STAGE1)
lg1 = infer(STAGE1, va, stages.CLASSES_STAGE1)
p1 = softmax(lg1.astype(np.float64) / T1)[:, 1]
flag = p1 >= THR
n_fa = int((flag & (y1 == 0)).sum())
print(f"[1단계] T={T1:.4f} · 임계값 {THR:.4f}  ({time.time()-t0:.0f}s)")
print(f"  넘긴 사진 {int(flag.sum()):,}장 — 그중 **헛알림 {n_fa:,}장** "
      f"({n_fa / max(int(flag.sum()), 1):.1%})")
print(f"  recall {float(flag[y1 == 1].mean()):.3f} · "
      f"헛알림률 {float(flag[y1 == 0].mean()):.1%}")


## 4. 2단계 — 넘긴 사진 **전부**에 이름을 붙여봅니다

헛알림(멀쩡한 개)에도 붙입니다. 그게 실제로 일어나는 일이니까요.


In [ ]:
sub = va[flag].reset_index(drop=True)
P2, names = {}, []
for name, exp, tag in ARMS:
    frame = crop.switch_tag(sub, tag, verbose=False)
    T2 = temperature(exp)
    lg = infer(exp, frame, CLASSES)
    P2[name] = softmax(lg.astype(np.float64) / T2)
    names.append(name)
    print(f"[2단계] {name:16} 완료 ({time.time()-t0:.0f}s)")

truth = sub["label_orig"].to_numpy()
out = Path("/kaggle/working/step31_arrays_full.npz")
np.savez_compressed(out, p1=p1[flag], truth=truth,
                    **{f"p2_{i}": P2[n] for i, n in enumerate(names)},
                    arm_names=np.array(names, dtype=object))
print(f"\n★ 저장: {out}  ← **이 파일 하나만 받으면 됩니다**")
print(f"   {out.stat().st_size / 1e6:.1f} MB · {len(truth):,}행")


## 5. 판정 — 기준은 `src/` 에 이미 박혀 있습니다

`experiments.naming_report` · `granularity_report` 를 그대로 부릅니다.
노트북에서 기준을 새로 쓰지 않습니다 (작업 규칙 3).


In [ ]:
from src import experiments
from src.config import MORPH_GROUP, MORPH_GROUP_KEEP_A6, URGENCY_TIER

ens = np.mean([P2[n] for n in names], 0)
is_norm = truth == stages.NORMAL_LABEL
p1s = p1[flag]
tier_true = np.array([URGENCY_TIER.get(t, -1) for t in truth])
a6 = CLASSES.index("A6")

print(f"1단계가 넘긴 사진 {len(truth):,}장 (헛알림 {is_norm.mean():.1%} 포함)\n")
said = np.array(CLASSES, dtype=object)[ens.argmax(1)]
res = {"6종 이름": experiments.naming_report(
    {"conf": p1s * ens.max(1), "wrong": is_norm | (said != truth),
     "is_a6": truth == "A6", "said_a6": ens.argmax(1) == a6})}

print()
for label, mp in (("계열 4군 (A6 따로)", MORPH_GROUP_KEEP_A6),
                  ("계열 3군", MORPH_GROUP),
                  ("긴급도 3등급", {k: str(v) for k, v in URGENCY_TIER.items()})):
    keys = list(dict.fromkeys(mp[c] for c in CLASSES))
    g = np.stack([ens[:, [i for i, c in enumerate(CLASSES) if mp[c] == k]].sum(1)
                  for k in keys], 1)
    gi = g.argmax(1)
    sd = np.array(keys, dtype=object)[gi]
    tr = np.array([mp.get(t, "정상") for t in truth], dtype=object)
    ts = np.array([max(URGENCY_TIER[c] for c in CLASSES if mp[c] == k)
                   for k in keys])[gi]
    res[label] = experiments.granularity_report(label, {
        "conf": p1s * g.max(1), "wrong": is_norm | (sd != tr),
        "tier_true": tier_true, "tier_said": ts})

# VL01 에서 잰 값과 나란히 — **방향을 짐작하지 않기 위해**
print("\n" + "=" * 66)
print("VL01(STEP 27~30) 과 나란히 — 커버리지, 오답률 20% 목표")
print("=" * 66)
VL01 = {"6종 이름": 0.337, "계열 4군 (A6 따로)": 0.614, "계열 3군": 0.621,
        "긴급도 3등급": 0.427}
print(f"  {'알갱이':22}{'VL01':>10}{'전체':>10}{'차이':>10}")
for k, v in VL01.items():
    now = res[k]["coverage"]
    print(f"  {k:22}{v:>10.1%}{now:>10.1%}{now - v:>+10.1%}")

j = Path("/kaggle/working/step31_naming_full.json")
j.write_text(json.dumps(
    {"step": "STEP 31 — 헛알림 포함 커버리지 (전체 데이터)", "chunks": keep,
     "n_stage1_val": int(len(va)), "n_flagged": int(flag.sum()),
     "n_false_alarm": n_fa, "stage1_threshold": THR,
     "stage1_recall": float(flag[y1 == 1].mean()),
     "stage1_false_alarm_rate": float(flag[y1 == 0].mean()),
     "results": res, "vl01_for_comparison": VL01},
    indent=2, ensure_ascii=False, default=float), encoding="utf-8")
print(f"\n저장: {j}")


## 받아야 할 것

| 파일 | 왜 |
|---|---|
| `step31_arrays_full.npz` | ★ **이거 하나면 나머지 분석이 전부 로컬에서 돕니다** |
| `step31_naming_full.json` | 판정 요약 |

로컬에서:

```bash
uv run --extra train python tools/naming_granularity.py \
    --arrays <받은 경로>/step31_arrays_full.npz \
    --out data/work/reports/step31_granularity_full.json
```

⚠️ **VL01 값과 섞지 마세요.** 같은 지표라도 분모가 다릅니다 —
전체 데이터의 헛알림 비율이 VL01(20.8%)과 다르면 커버리지도 따라 움직입니다.
그 비율이 위 셀 3 에 찍힙니다.
